In [1]:
import subprocess, sys, os

from google.colab import drive
drive.mount('/content/drive')

# Системні залежності
subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libsndfile1'])

# Python-пакети
def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('git+https://github.com/pyannote/pyannote-audio.git@main')
pip('pyannote.metrics', 'pyannote.database', 'pyannote.pipeline')
pip('lightning')
pip('speechbrain', 'librosa', 'soundfile')
pip('huggingface_hub==1.9.0')

# Службова директорія для database.yml
os.makedirs('AMI-diarization-setup/pyannote', exist_ok=True)

# HuggingFace токен
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab Secrets.')
except Exception:
    os.environ['HF_TOKEN'] = '-------------'
    print('HF_TOKEN set manually — replace with your real token.')

print('All done.')


Mounted at /content/drive
HF_TOKEN loaded from Colab Secrets.
All done.


In [ ]:
# 0. Imports & global config
import os, random
from pathlib import Path
from typing import Iterable, List

import numpy as np
import torch

from pyannote.database import registry, FileFinder
from pyannote.metrics.diarization import DiarizationErrorRate

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

HF_TOKEN = os.environ.get('HF_TOKEN', '-----------------------')

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
print('Device:', DEVICE)

ROOT = Path('/content')

DATASET_DIR   = Path('/content/drive/MyDrive/Colab Notebooks/audio_lab2/data_voxconverse')

DEV_WAV_DIR   = DATASET_DIR / 'dev'  / 'wav'
DEV_RTTM_DIR  = DATASET_DIR / 'dev'  / 'rttm'
TEST_WAV_DIR  = DATASET_DIR / 'test' / 'wav'
TEST_RTTM_DIR = DATASET_DIR / 'test' / 'rttm'
AMI_DIR       = ROOT / 'AMI-diarization-setup' / 'pyannote'

print('DATASET_DIR =', DATASET_DIR)
assert DATASET_DIR.exists(), f'Dataset not found: {DATASET_DIR}'


Device: cuda
DATASET_DIR = /content/drive/MyDrive/Colab Notebooks/audio_lab2/data_voxconverse


In [3]:
# 1. Build train/val/test lists, UEM files, database.yml

def file_stem_list(path: Path, ext: str = '.wav') -> List[str]:
    return sorted(p.stem for p in path.glob(f'*{ext}'))

def write_list(path: Path, items: Iterable[str]):
    path.write_text('\n'.join(items) + '\n', encoding='utf-8')

def build_uem(wav_dir: Path, uris: List[str], uem_dir: Path):
    import wave as _wave
    uem_dir.mkdir(parents=True, exist_ok=True)
    for uri in uris:
        with _wave.open(str(wav_dir / f'{uri}.wav'), 'rb') as wf:
            dur = wf.getnframes() / wf.getframerate()
        (uem_dir / f'{uri}.uem').write_text(
            f'{uri} 1 0.000 {dur:.3f}\n', encoding='utf-8'
        )

def write_database_yml(
    path: Path,
    dev_wav: Path, test_wav: Path,
    train_lst: Path, val_lst: Path, test_lst: Path,
    train_uem: Path, val_uem: Path, test_uem: Path,
):
    txt = f"""Databases:
  VOXCONVERSE:
    - {dev_wav.as_posix()}/{{uri}}.wav
    - {test_wav.as_posix()}/{{uri}}.wav

Protocols:
  VOXCONVERSE:
    SpeakerDiarization:
      custom:
        scope: file
        train:
          uri:        {train_lst.as_posix()}
          annotation: {DEV_RTTM_DIR.as_posix()}/{{uri}}.rttm
          annotated:  {train_uem.as_posix()}/{{uri}}.uem
        development:
          uri:        {val_lst.as_posix()}
          annotation: {DEV_RTTM_DIR.as_posix()}/{{uri}}.rttm
          annotated:  {val_uem.as_posix()}/{{uri}}.uem
        test:
          uri:        {test_lst.as_posix()}
          annotation: {TEST_RTTM_DIR.as_posix()}/{{uri}}.rttm
          annotated:  {test_uem.as_posix()}/{{uri}}.uem
"""
    path.write_text(txt, encoding='utf-8')

# collect URIs
dev_uris  = file_stem_list(DEV_WAV_DIR)
test_uris = file_stem_list(TEST_WAV_DIR)
assert dev_uris  and test_uris, 'WAV files not found'

# 80/20 split of dev -> train / val
random.seed(SEED)
shuffled  = dev_uris[:]
random.shuffle(shuffled)
split     = max(1, int(0.8 * len(shuffled)))
train_uris, val_uris = sorted(shuffled[:split]), sorted(shuffled[split:])

lists_dir = DATASET_DIR / 'lists'
lists_dir.mkdir(parents=True, exist_ok=True)
train_lst = lists_dir / 'train.txt'
val_lst   = lists_dir / 'val.txt'
test_lst  = lists_dir / 'test.txt'
write_list(train_lst, train_uris)
write_list(val_lst,   val_uris)
write_list(test_lst,  test_uris)

train_uem = DATASET_DIR / 'train' / 'uem'
val_uem   = DATASET_DIR / 'val'   / 'uem'
test_uem  = DATASET_DIR / 'test'  / 'uem'
build_uem(DEV_WAV_DIR,  train_uris, train_uem)
build_uem(DEV_WAV_DIR,  val_uris,   val_uem)
build_uem(TEST_WAV_DIR, test_uris,  test_uem)

AMI_DIR.mkdir(parents=True, exist_ok=True)
db_yml = AMI_DIR / 'database.yml'
write_database_yml(db_yml, DEV_WAV_DIR, TEST_WAV_DIR,
                   train_lst, val_lst, test_lst,
                   train_uem, val_uem, test_uem)

os.environ['PYANNOTE_DATABASE_CONFIG'] = str(db_yml)
print('database.yml written:', db_yml)
print(f'train={len(train_uris)}  val={len(val_uris)}  test={len(test_uris)}')

database.yml written: /content/AMI-diarization-setup/pyannote/database.yml
train=172  val=44  test=232


In [4]:
# 2. Load pyannote protocol
from pyannote.audio import Pipeline

registry.load_database(os.environ['PYANNOTE_DATABASE_CONFIG'])
protocol = registry.get_protocol(
    'VOXCONVERSE.SpeakerDiarization.custom',
    preprocessors={'audio': FileFinder()}
)
print('Protocol loaded.')

Protocol loaded.


In [6]:
# 3. Baseline DER (pretrained community-1 pipeline)
from tqdm import tqdm

baseline_pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-community-1',
    token=HF_TOKEN
).to(DEVICE)

test_files = list(protocol.test())
metric_base = DiarizationErrorRate()
print("Started")
idx = 0
for file in tqdm(test_files, desc='Baseline DER'):
    output     = baseline_pipeline(file)
    hypothesis = output.speaker_diarization
    metric_base(file['annotation'], hypothesis, uem=file['annotated'])
    idx += 1
    if idx%10 == 0:
        print(f"DER for {idx} files is: {100*abs(metric_base)}")


baseline_der = 100 * abs(metric_base)
print(f'Baseline DER: {baseline_der:.2f}%')

Started


Baseline DER:   0%|          | 0/232 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  std = sequences.std(dim=-1, correction=1)
Baseline DER:   1%|          | 2/232 [00:31<1:00:49, 15.87s/it]/usr/local/lib/python3.12/dist-packages/pyannote/database/protocol/protocol

DER for 10 files is: 14.116782679180012


/usr/local/lib/python3.12/dist-packages/pyannote/database/protocol/protocol.py:140: UserWarning: Existing precomputed key "annotation" has been modified by a preprocessor.
  warnings.warn(msg.format(key=key))
Baseline DER:   6%|▌         | 14/232 [12:10<2:16:57, 37.69s/it]/usr/local/lib/python3.12/dist-packages/pyannote/database/protocol/protocol.py:140: UserWarning: Existing precomputed key "annotation" has been modified by a preprocessor.
  warnings.warn(msg.format(key=key))
Baseline DER:   6%|▋         | 15/232 [12:22<1:48:24, 29.97s/it]/usr/local/lib/python3.12/dist-packages/pyannote/database/protocol/protocol.py:140: UserWarning: Existing precomputed key "annotation" has been modified by a preprocessor.
  warnings.warn(msg.format(key=key))
Baseline DER:   8%|▊         | 19/232 [15:53<2:13:25, 37.58s/it]/usr/local/lib/python3.12/dist-packages/pyannote/database/protocol/protocol.py:140: UserWarning: Existing precomputed key "annotation" has been modified by a preprocessor.
  warning

DER for 20 files is: 11.370818834341003


...

DER for 90 files is: 11.50554439197975


Baseline DER:  39%|███▉      | 91/232 [1:31:57<2:59:23, 76.33s/it]/usr/local/lib/python3.12/dist-packages/pyannote/database/protocol/protocol.py:140: UserWarning: Existing precomputed key "annotation" has been modified by a preprocessor.
  warnings.warn(msg.format(key=key))
Baseline DER:  40%|███▉      | 92/232 [1:32:03<2:08:25, 55.04s/it]/usr/local/lib/python3.12/dist-packages/pyannote/database/protocol/protocol.py:140: UserWarning: Existing precomputed key "annotation" has been modified by a preprocessor.
  warnings.warn(msg.format(key=key))
Baseline DER:  41%|████      | 94/232 [1:35:31<3:00:56, 78.67s/it]/usr/local/lib/python3.12/dist-packages/pyannote/database/protocol/protocol.py:140: UserWarning: Existing precomputed key "annotation" has been modified by a preprocessor.
  warnings.warn(msg.format(key=key))
Baseline DER:  43%|████▎     | 99/232 [1:39:55<1:57:07, 52.84s/it]/usr/local/lib/python3.12/dist-packages/pyannote/database/protocol/protocol.py:140: UserWarning: Existing pre

DER for 100 files is: 12.072416138030855


Baseline DER:  43%|████▎     | 100/232 [1:43:25<2:16:31, 62.06s/it]


KeyboardInterrupt: 

In [7]:
# 4. Fine-tune the segmentation model that is part of community-1
from types import MethodType
from torch.optim import Adam
from pyannote.audio import Model
from pyannote.audio.tasks import SpeakerDiarization as SegmentationTask
from pyannote.audio.pipelines import SpeakerDiarization
from pyannote.pipeline import Optimizer
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
import time

def section(title):
    print(f'\n{"─"*60}')
    print(f'  {title}')
    print(f'{"─"*60}')

# Load pretrained segmentation model
section('STEP 1/3 · Loading pretrained segmentation model')
SEG_CACHE = Path(
    '~/.cache/huggingface/hub'
    '/models--pyannote--speaker-diarization-community-1'
    '/snapshots/3533c8cf8e369892e6b79ff1bf80f7b0286a54ee'
    '/segmentation/pytorch_model.bin'
).expanduser()
assert SEG_CACHE.exists(), f'Segmentation model not found: {SEG_CACHE}'
model = Model.from_pretrained(str(SEG_CACHE))
print(f'Model loaded from: {SEG_CACHE}')

# Set up training task
section('STEP 2/3 · Setting up training task & callbacks')
task = SegmentationTask(
    protocol,
    duration=model.specifications.duration,
    max_speakers_per_chunk=len(model.specifications.classes),
    batch_size=16,
    num_workers=2,
)
model.task = task

def configure_optimizers(self):
    return Adam(self.parameters(), lr=1e-4)
model.configure_optimizers = MethodType(configure_optimizers, model)

monitor, direction = task.val_monitor
print(f'Monitoring: {monitor}  (direction: {direction})')

checkpoint = ModelCheckpoint(
    monitor=monitor, mode=direction,
    save_top_k=1, every_n_epochs=1, filename='{epoch}',
)
early_stopping = EarlyStopping(monitor=monitor, mode=direction, patience=5)
print('EarlyStopping patience=5 epochs')

# Train
section('STEP 3/3 · Training (max 10 epochs, early stop on val DER)')
t0 = time.time()
trainer = Trainer(
    accelerator=DEVICE.type,
    max_epochs=10,
    callbacks=[checkpoint, early_stopping],
    gradient_clip_val=0.5,
)
trainer.fit(model)
elapsed = time.time() - t0

finetuned_ckpt = checkpoint.best_model_path
print(f'\nTraining done in {elapsed/60:.1f} min')
print(f'Best checkpoint: {finetuned_ckpt}')

Epoch 9/9  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367/367 0:03:28 • 0:00:00 1.79it/s v_num: 0.000 DiarizationErrorRate:
                                                                                 0.058                             
                                                                                 DiarizationErrorRate/Confusion:   
                                                                                 0.019                             
                                                                                 DiarizationErrorRate/DetectionErr…
                                                                                 0.039                             
                                                                                 DiarizationErrorRate/FalseAlarm:  
                                                                                 0.020 DiarizationErrorRate/Miss:  
                                                                                 0.019                             
                                                                                 DiarizationErrorRate/Precision:   
                                                                                 0.981 DiarizationErrorRate/Recall:
                                                                                 0.962                             

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.



Training done in 45.1 min
Best checkpoint: /content/lightning_logs/version_0/checkpoints/epoch=9.ckpt


In [8]:
# 5. Threshold tuning on dev split + final evaluation on test
import glob, time

# Load best checkpoint
section('STEP 1/3 · Loading best checkpoint')
_ckpts = sorted(glob.glob('lightning_logs/**/checkpoints/*.ckpt', recursive=True))
if _ckpts:
    finetuned_ckpt = _ckpts[-1]
    print(f'Checkpoint: {finetuned_ckpt}')
else:
    raise FileNotFoundError('No checkpoint found — run the fine-tuning cell first')

# Build fine-tuned pipeline
section('STEP 2/3 · Tuning clustering thresholds on dev set')
dev_set = list(protocol.development())
print(f'Dev set: {len(dev_set)} files')

pipe_ft = SpeakerDiarization(
    segmentation=finetuned_ckpt,
    embedding=baseline_pipeline.embedding,
    plda=baseline_pipeline.plda,
    clustering='VBxClustering',
    token=HF_TOKEN,
).to(DEVICE)
pipe_ft.instantiate(baseline_pipeline.parameters(instantiated=True))
pipe_ft.freeze({'segmentation': {'min_duration_off': 0.0}})

_orig_apply = pipe_ft.__class__.apply
def _apply_annotation(self, file, **kwargs):
    result = _orig_apply(self, file, **kwargs)
    return result.speaker_diarization if hasattr(result, 'speaker_diarization') else result
pipe_ft.apply = MethodType(_apply_annotation, pipe_ft)

print('Running Optuna optimization (up to 21 trials)...')
t0 = time.time()
optimizer = Optimizer(pipe_ft)
for i, result in enumerate(optimizer.tune_iter(dev_set, show_progress=True)):
    print(f'  trial [{i:02d}/20]  loss={result["loss"]:.4f}  params={result["params"]}')
    if i >= 20:
        break
print(f'Tuning done in {(time.time()-t0)/60:.1f} min')

pipe_ft.instantiate(optimizer.best_params)
print(f'Best params: {optimizer.best_params}')

# Evaluate on test
section('STEP 3/3 · Evaluating fine-tuned pipeline on test set')
protocol_test = list(protocol.test())
print(f'Test set: {len(protocol_test)} files\n')

metric_ft = DiarizationErrorRate()
t0 = time.time()
for idx, file in enumerate(tqdm(protocol_test, desc='Fine-tuned DER'), 1):
    hypothesis = pipe_ft(file)
    metric_ft(file['annotation'], hypothesis, uem=file['annotated'])
    if idx % 20 == 0:
        elapsed = time.time() - t0
        remaining = elapsed / idx * (len(protocol_test) - idx)
        print(f'  [{idx}/{len(protocol_test)}]  running DER={100*abs(metric_ft):.2f}%'
              f'  ~{remaining/60:.0f} min left')

finetuned_der = 100 * abs(metric_ft)
print(f'\n{"═"*60}')
print(f'  Fine-tuned DER: {finetuned_der:.2f}%')
print(f'  Baseline DER  : {baseline_der:.2f}%')
print(f'  Improvement   : {baseline_der - finetuned_der:+.2f}%')
print(f'{"═"*60}')


────────────────────────────────────────────────────────────
  STEP 1/3 · Loading best checkpoint
────────────────────────────────────────────────────────────
Checkpoint: lightning_logs/version_0/checkpoints/epoch=9.ckpt

────────────────────────────────────────────────────────────
  STEP 2/3 · Tuning clustering thresholds on dev set
────────────────────────────────────────────────────────────
Dev set: 44 files


/usr/local/lib/python3.12/dist-packages/pyannote/pipeline/parameter.py:160: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  return trial.suggest_uniform(name, self.low, self.high)


Running Optuna optimization (up to 21 trials)...



Current trial: 100%|██████████| 44/44 [20:45<00:00, 37.91s/it]
                                                              /usr/local/lib/python3.12/dist-packages/pyannote/pipeline/parameter.py:160: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  return trial.suggest_uniform(name, self.low, self.high)


  trial [00/20]  loss=0.4143  params={'segmentation': {'min_duration_off': 0.0}, 'clustering': {'threshold': 0.5510432032705158, 'Fa': 0.02093253628131773, 'Fb': 4.590036097831485}}


...

  [20/232]  running DER=9.94%  ~203 min left


Fine-tuned DER:  10%|▉         | 23/232 [21:53<3:08:38, 54.16s/it]/usr/local/lib/python3.12/dist-packages/pyannote/database/protocol/protocol.py:140: UserWarning: Existing precomputed key "annotation" has been modified by a preprocessor.
  warnings.warn(msg.format(key=key))
Fine-tuned DER:  11%|█         | 26/232 [27:40<5:39:57, 99.02s/it]/usr/local/lib/python3.12/dist-packages/pyannote/database/protocol/protocol.py:140: UserWarning: Existing precomputed key "annotation" has been modified by a preprocessor.
  warnings.warn(msg.format(key=key))
Fine-tuned DER:  15%|█▌        | 35/232 [35:45<2:33:22, 46.71s/it]/usr/local/lib/python3.12/dist-packages/pyannote/database/protocol/protocol.py:140: UserWarning: Existing precomputed key "annotation" has been modified by a preprocessor.
  warnings.warn(msg.format(key=key))
Fine-tuned DER:  16%|█▋        | 38/232 [36:48<1:31:17, 28.23s/it]/usr/local/lib/python3.12/dist-packages/pyannote/database/protocol/protocol.py:140: UserWarning: Existing pre

  [40/232]  running DER=11.27%  ~195 min left


Fine-tuned DER:  19%|█▉        | 44/232 [42:09<3:00:09, 57.50s/it]


KeyboardInterrupt: 

In [10]:
baseline_der = 12
finetuned_der = 100 * abs(metric_ft)
print(f'\n{"═"*60}')
print(f'  Fine-tuned DER: {finetuned_der:.2f}%')
print(f'  Baseline DER  : {baseline_der:.2f}%')
print(f'  Improvement   : {baseline_der - finetuned_der:+.2f}%')
print(f'{"═"*60}')


════════════════════════════════════════════════════════════
  Fine-tuned DER: 11.33%
  Baseline DER  : 12.00%
  Improvement   : +0.67%
════════════════════════════════════════════════════════════
